# llama — does more compute break the plateau?

The main study gave every model the same small budget: 2,000 rows per client per
round, 10 rounds. llama reached macro F1 **0.449** (IID) and **0.417** (non-IID)
— but it stopped improving early:

| | round 2 | round 10 | gain over 8 rounds |
|---|---|---|---|
| IID | 0.4199 | 0.4493 | **+0.029** |
| non-IID (round 4 → 10) | 0.3869 | 0.4168 | **+0.030** |

The loss stopped falling too, oscillating between 1.20 and 1.46 from round 3 on.
This notebook spends **4.8× the compute** on the same model to find out whether
that plateau is a budget limit or a real ceiling.

**What changes, and why it is rows rather than rounds.** Each round runs the
learning rate from 0 up to 2e-4 and back to 0 over just 125 steps, with a fresh
optimiser every time — so every round is a tiny training run that never sustains
a useful rate. Adding rounds multiplies that waste. Adding rows per round
stretches the schedule instead, giving the model a long descent per round.

| | baseline | this run |
|---|---|---|
| rows / client / round | 2,000 | **8,000** |
| rounds | 10 | **12** |
| steps / client / round | 125 | **500** |
| total steps / client | 1,250 | **6,000** |
| total rows / client | 20,000 | 96,000 |
| wall clock | ~32 min | **~2.6 h** |

96,000 is deliberate: the smallest non-IID shard holds 98,616 rows, so staying
under it means every client trains on fresh rows the whole way rather than
wrapping back to the start.

Everything else — learning rate, LoRA rank, batch size, aggregation — is left
alone, so the result answers one question and not two.

**Setup:** Accelerator `GPU T4 x2` · Internet `On` · NF-ToN-IoT attached ·
`HF_TOKEN` in Add-ons → Secrets. Both splits run at once, one per GPU, so the
whole thing finishes in the time one of them takes.

## 1. Environment

In [ ]:
!pip install -q -U transformers peft accelerate comet_ml
# Kaggle ships torchao 0.10; peft's LoRA dispatcher raises on anything below 0.16
# rather than skipping it. Nothing here uses torchao quantisation.
!pip uninstall -q -y torchao

import torch
print('torch', torch.__version__, '| GPUs:', torch.cuda.device_count())

## 2. Code and data

Delete `/kaggle/working/repo` first if you have pushed changes since the last run
— the clone is skipped when the directory already exists.

In [ ]:
import glob, os

BRANCH = 'llm'
REPO = '/kaggle/working/repo'

if not os.path.exists(REPO):
    !git clone -q -b {BRANCH} https://github.com/asfi50/Fed_GNN.git {REPO}
os.chdir(REPO)
!git log -1 --oneline

CSV = next(c for c in glob.glob('/kaggle/input/**/*.csv', recursive=True)
           if 'NF-ToN-IoT' in c and not any(x in c for x in ('common', 'easy', 'uncommon')))
DATA, RESULTS = '/kaggle/working/dataset', '/kaggle/working/results'
print('dataset :', CSV)

In [ ]:
from kaggle_secrets import UserSecretsClient

os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
print('HF_TOKEN loaded:', bool(os.environ.get('HF_TOKEN')))

## 3. Shards

Identical to the baseline run — same seed, same Dirichlet alpha, so the
partition is byte-for-byte the one the 32-minute result was measured on. Only the
per-round budget differs.

In [ ]:
!python llm/split_dataset.py --input_file {CSV} --output_dir {DATA} --num_clients 5 --alpha 0.5

## 4. Pre-flight

Two minutes to confirm llama still loads, attaches LoRA, takes a batch and
aggregates — worth it before committing three hours of quota.

In [ ]:
!python llm/check_models.py --forward --verbose --models llama

## 5. The run

`--tag long` keeps this separate from the baseline in Comet, on disk and on the
Hub; without it both would be called `llama_iid_naive` and the comparison table
would silently mix them. `--no_push` leaves the baseline adapter on the Hub
untouched.

Full logs are written to `/kaggle/working/logs/`, which is saved as notebook
output. Progress prints every three minutes.

**If the session dies**, add `'--resume'` to `ARGS` and rerun this cell — every
round is checkpointed.

In [ ]:
import subprocess, time, datetime

LOGS = '/kaggle/working/logs'
os.makedirs(LOGS, exist_ok=True)

ARGS = ['--model', 'llama', '--rows_per_round', '8000', '--rounds', '12',
        '--tag', 'long', '--no_push',
        '--data_dir', DATA, '--output_dir', RESULTS]

procs = {}
for gpu, split in [(0, 'iid'), (1, 'non_iid')]:
    path = f'{LOGS}/llama_{split}_long.log'
    fh = open(path, 'w')
    procs[split] = (subprocess.Popen(
        ['python', '-u', 'llm/run_experiment.py', '--split', split] + ARGS,
        env={**os.environ, 'CUDA_VISIBLE_DEVICES': str(gpu)},
        stdout=fh, stderr=subprocess.STDOUT), fh, path)
    print(f'{split:8} -> GPU {gpu}   {path}')

started = time.time()
while any(p.poll() is None for p, _, _ in procs.values()):
    time.sleep(180)
    print(f'\n[{datetime.datetime.now():%H:%M}  +{(time.time()-started)/60:.0f} min]')
    for split, (p, _, path) in procs.items():
        hits = subprocess.run(['grep', '-E', 'complete|VAL', path],
                              capture_output=True, text=True).stdout.strip().splitlines()
        state = 'running' if p.poll() is None else f'exit {p.returncode}'
        print(f'  {split:8} [{state}] {hits[-1][:115] if hits else "loading model..."}')

for split, (p, fh, path) in procs.items():
    fh.close()
    print(f'\n{"="*70}\n  {split}   exit code {p.returncode}\n{"="*70}')
    print(subprocess.run(['tail', '-30', path], capture_output=True, text=True).stdout)

## 6. Did it help?

Pulls the baseline straight from Comet, so the comparison works even though those
runs happened in an earlier session whose `/kaggle/working` is long gone.

In [ ]:
import sys, json
import numpy as np
import pandas as pd
from comet_ml import API

sys.path.insert(0, 'llm')
from fedllm.tracking import COMET_API_KEY, PROJECT_NAME

EXCLUDED = ('mitm', 'ransomware')
api = API(api_key=COMET_API_KEY)
exps = {e.name: e for e in api.get('text-films', PROJECT_NAME)}

rows = []
for split in ['iid', 'non_iid']:
    for label, name in [('baseline', f'llama_{split}_naive'),
                        ('long', f'llama_{split}_naive_long')]:
        e = exps.get(name)
        asset = e and next((a['assetId'] for a in e.get_asset_list()
                            if a.get('fileName') == 'results.json'), None)
        if not asset:
            rows.append({'split': split, 'run': label, 'status': 'missing'})
            continue
        d = json.loads(e.get_asset(asset))
        t = d['test_metrics']
        rows.append({
            'split': split, 'run': label, 'status': 'ok',
            'rows/round': d['rows_per_client_per_round'], 'rounds': d['num_rounds'],
            'accuracy': round(t['accuracy'], 4),
            'balanced_acc': round(t['balanced_accuracy'], 4),
            'macro_f1': round(t['macro_f1'], 4),
            'macro_f1_8': round(float(np.mean([v for k, v in t['per_class_f1'].items()
                                               if k not in EXCLUDED])), 4),
            'minutes': round(d['total_seconds'] / 60),
        })

df = pd.DataFrame(rows)
display(df)

for split in ['iid', 'non_iid']:
    sel = df[(df.split == split) & (df.status == 'ok')].set_index('run')
    if {'baseline', 'long'} <= set(sel.index):
        b, l = sel.loc['baseline', 'macro_f1_8'], sel.loc['long', 'macro_f1_8']
        print(f'{split:8} macro F1* {b:.4f} -> {l:.4f}   ({100*(l-b)/b:+.1f} % for 4.8x the compute)')

## Reading the outcome

**A clear jump** — say macro F1\* past 0.60 — means the plateau was a budget
limit, and the other four models deserve the same budget before any of the
cross-model claims stand.

**Barely moved** — within a point or two — means the ceiling is not compute. The
per-class analysis already points at why: scanning, injection, xss and password
collapse into one another, and ten flow fields cannot tell them apart. That
finding makes the case for the GNN branch, where the graph carries exactly the
signal these features lack.

Either way the next experiment is the same one: hold the budget and flatten the
learning-rate sawtooth, so the schedule stops restarting from zero every round.